# Data Preprocessing

# Movie-Movie Connections

Filter data

In [ ]:
#filter ratings<min_rating
#for limit 3 & 4

import csv

def filter_ratings(input_file, output_file, min_rating=4.0):
    # Store filtered rows
    filtered_rows = []

    with open(input_file, 'r') as infile:
        reader = csv.DictReader(infile)
        # Verify expected columns
        if 'Weight' not in reader.fieldnames:
            raise ValueError("No 'rating' column found in the input file.")

        # Keep header
        header = reader.fieldnames
        filtered_rows.append(header)

        # Filter rows
        for row in reader:
            try:
                rating = float(row['Weight'])  # Convert rating to float
                if rating >= min_rating:
                    filtered_rows.append([row[field] for field in header])
            except ValueError:
                print(f"Skipping invalid rating: {row['Weight']} in row {row}")

    # Write filtered data to new file
    with open(output_file, 'w', newline='') as outfile:
        writer = csv.writer(outfile)
        writer.writerows(filtered_rows)

    print(f"Filtered ratings saved to {output_file}. Rows kept: {len(filtered_rows) - 1}")

# Usage
input_file = 'ratings.csv'
output_file = 'filtered_ratings.csv'
filter_ratings(input_file, output_file, min_rating=4.0)

Projection of movies

In [ ]:
# import csv
# from collections import defaultdict
# from itertools import combinations

# # Minimum number of shared users required for an edge
# MIN_SHARED_USERS = 25

# # Step 1: Parse the user-movie data and group movies by user
# def load_user_movie_data(file_path, user_col='Source', movie_col='Target'):
#     user_to_movies = defaultdict(list)
#     unique_movies = set()
#     with open(file_path, 'r', encoding='utf-8-sig') as file:
#         reader = csv.DictReader(file)
#         if user_col not in reader.fieldnames or movie_col not in reader.fieldnames:
#             raise KeyError(f"Columns '{user_col}' or '{movie_col}' not found in {file_path}. Available columns: {reader.fieldnames}")
#         for row in reader:
#             user = row[user_col]
#             movie = row[movie_col]
#             user_to_movies[user].append(movie)
#             unique_movies.add(movie)
#     print(f"Number of unique movies: {len(unique_movies)}")
#     return user_to_movies

# # Step 2: Create the movie-movie projection with a minimum threshold
# def create_movie_projection(user_to_movies, min_shared_users=MIN_SHARED_USERS):
#     movie_pairs = defaultdict(int)
#     for user, movies in user_to_movies.items():
#         movies = sorted(set(movies))  # Remove duplicates per user
#         for movie1, movie2 in combinations(movies, 2):
#             movie_pairs[(movie1, movie2)] += 1

#     # Filter pairs with at least min_shared_users
#     filtered_pairs = {pair: count for pair, count in movie_pairs.items() if count >= min_shared_users}
#     return filtered_pairs

# # Step 3: Write the projection to a CSV file and check for multi-edges
# def write_projection_to_csv(movie_pairs, output_file):
#     seen_pairs = set()
#     with open(output_file, 'w', newline='') as file:
#         writer = csv.writer(file)
#         writer.writerow(['Source', 'Target', 'Weight'])
#         for (movie1, movie2), weight in movie_pairs.items():
#             pair = (movie1, movie2)
#             if pair in seen_pairs:
#                 print(f"Warning: Duplicate edge detected: {movie1},{movie2}")
#             else:
#                 seen_pairs.add(pair)
#                 writer.writerow([movie1, movie2, weight])
#     print(f"Number of unique edges in projection (with >= {MIN_SHARED_USERS} shared users): {len(seen_pairs)}")

# # Main function
# def main(input_file, output_file, user_col='Source', movie_col='Target'):
#     print("Loading user-movie data...")
#     user_to_movies = load_user_movie_data(input_file, user_col, movie_col)

#     print("Creating movie-movie projection...")
#     movie_pairs = create_movie_projection(user_to_movies)

#     print("Writing projection to CSV...")
#     write_projection_to_csv(movie_pairs, output_file)
#     print(f"Projection complete. Output saved to {output_file}")

# # Example usage
# if __name__ == "__main__":
#     input_file = 'filtered_ratings.csv'  # Your input file path
#     output_file = 'ratings_projection.csv'     # Output file path

#     user_column = 'Source'  # Matches your file with BOM
#     movie_column = 'Target'       # Matches your file

#     main(input_file, output_file, user_col=user_column, movie_col=movie_column)

import csv
from collections import defaultdict
from itertools import combinations

# Threshold to determine like/dislike
LIKE_THRESHOLD = 3.0

# Minimum number of shared users to consider the edge
MIN_SHARED_USERS = 5

def load_user_movie_ratings(file_path, user_col='userId', movie_col='movieId', rating_col='rating'):
    user_to_movies = defaultdict(dict)
    unique_movies = set()

    with open(file_path, 'r', encoding='utf-8-sig') as file:
        reader = csv.DictReader(file)
        if user_col not in reader.fieldnames or movie_col not in reader.fieldnames or rating_col not in reader.fieldnames:
            raise KeyError(f"Required columns not found. Found: {reader.fieldnames}")

        for row in reader:
            user = row[user_col]
            movie = row[movie_col]
            rating = float(row[rating_col])
            user_to_movies[user][movie] = rating
            unique_movies.add(movie)

    print(f"Number of unique movies: {len(unique_movies)}")
    return user_to_movies

def compute_signed_contribution(r1, r2, threshold=LIKE_THRESHOLD):
    if r1 >= threshold and r2 >= threshold:
        return 1
    elif r1 < threshold and r2 < threshold:
        return 0.5
    else:
        return -1

def create_signed_movie_projection(user_to_movies, min_shared_users=MIN_SHARED_USERS):
    movie_pairs_contributions = defaultdict(list)

    for user, movie_ratings in user_to_movies.items():
        movies = sorted(movie_ratings.keys())
        for movie1, movie2 in combinations(movies, 2):
            r1, r2 = movie_ratings[movie1], movie_ratings[movie2]
            contribution = compute_signed_contribution(r1, r2)
            movie_pairs_contributions[(movie1, movie2)].append(contribution)

    # Aggregate only if number of users is sufficient
    signed_weights = {
        pair: sum(contributions)
        for pair, contributions in movie_pairs_contributions.items()
        if len(contributions) >= min_shared_users
    }

    return signed_weights

def write_projection_to_csv(movie_pairs, output_file):
    with open(output_file, 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Source', 'Target', 'SignedWeight'])
        for (movie1, movie2), weight in movie_pairs.items():
            writer.writerow([movie1, movie2, weight])
    print(f"Projection complete. {len(movie_pairs)} edges written to {output_file}")

# Main function
def main(input_file, output_file, user_col='userId', movie_col='movieId', rating_col='rating'):
    print("Loading user-movie ratings...")
    user_to_movies = load_user_movie_ratings(input_file, user_col, movie_col, rating_col)

    print("Creating signed movie-movie projection...")
    signed_movie_pairs = create_signed_movie_projection(user_to_movies)

    print("Writing projection to CSV...")
    write_projection_to_csv(signed_movie_pairs, output_file)

# Example usage
if __name__ == "__main__":
    input_file = 'ratings.csv'  # Path to MovieLens ratings file
    output_file = 'signed_movie_projection.csv'

    main(input_file, output_file, user_col='userId', movie_col='movieId', rating_col='rating')


Loading user-movie ratings...
Number of unique movies: 9724
Creating signed movie-movie projection...
Writing projection to CSV...
Projection complete. 1293963 edges written to signed_movie_projection.csv


Extract unique movies

In [ ]:
import csv

def write_unique_movies_to_csv(input_file, output_file):
    unique_movies = set()
    with open(input_file, 'r') as infile:
        reader = csv.DictReader(infile)
        for row in reader:
            unique_movies.add(row['Source'])  # Add Source movie ID
            unique_movies.add(row['Target'])  # Add Target movie ID

    with open(output_file, 'w', newline='') as outfile:
        writer = csv.writer(outfile)
        writer.writerow(['Id'])  # Write header
        for movie in sorted(unique_movies, key=int):  # Sort as integers
            writer.writerow([movie])

# Usage
input_file = 'ratings_projection.csv'
output_file = 'filtered_movies.csv'
write_unique_movies_to_csv(input_file, output_file)
print(f"Unique movies saved to {output_file}")

Analyzing communties based on modularity class

In [ ]:
import pandas as pd

# Load the nodes with community assignments from Gephi
nodes_df = pd.read_csv('community_detection.csv')
# Remove 'm' prefix from movie IDs to match movies.csv
nodes_df['movieId'] = nodes_df['Id']

# Load the movies data with genres
movies_df = pd.read_csv('movies.csv')

# Merge the two dataframes on movieId
merged_df = pd.merge(nodes_df, movies_df, on='movieId', how='inner')

# Save the merged data for reference
merged_df.to_csv('movies_with_communities.csv', index=False)

# Display the first few rows to verify
print(merged_df[['movieId', 'title', 'genres', 'modularity_class', 'stat_inf_class']].head())

   movieId                           title  \
0        1                Toy Story (1995)   
1        2                  Jumanji (1995)   
2        6                     Heat (1995)   
3       10                GoldenEye (1995)   
4       11  American President, The (1995)   

                                        genres  modularity_class  \
0  Adventure|Animation|Children|Comedy|Fantasy                 1   
1                   Adventure|Children|Fantasy                 1   
2                        Action|Crime|Thriller                 0   
3                    Action|Adventure|Thriller                 1   
4                         Comedy|Drama|Romance                 1   

   stat_inf_class  
0               8  
1              21  
2              29  
3               9  
4               9  


Extracting genres

In [ ]:
from collections import Counter

# Function to split genres and count them per community
def analyze_genres_per_community(df):
    # Create a dictionary to store genre counts per community
    community_genres = {}

    # Iterate over each community
    for community_id in df['modularity_class'].unique():
        # Filter movies in this community
        community_movies = df[df['modularity_class'] == community_id]
        # Split genres and flatten the list
        genres_list = []
        for genres in community_movies['genres']:
            if pd.notna(genres) and genres != '(no genres listed)':
                genres_list.extend(genres.split('|'))
        # Count genre occurrences
        genre_counts = Counter(genres_list)
        community_genres[community_id] = genre_counts

    return community_genres

# Run the analysis
community_genres = analyze_genres_per_community(merged_df)

# Print results
for community_id, genre_counts in community_genres.items():
    print(f"Community {community_id} ({len(merged_df[merged_df['modularity_class'] == community_id])} movies):")
    print(genre_counts.most_common())
    print()

Community 1 (99 movies):
[('Drama', 55), ('Thriller', 33), ('Comedy', 32), ('Action', 25), ('Crime', 24), ('Adventure', 22), ('Romance', 17), ('Children', 11), ('Fantasy', 11), ('Animation', 8), ('Sci-Fi', 8), ('War', 7), ('Horror', 6), ('Musical', 6), ('Mystery', 5), ('IMAX', 3), ('Western', 3), ('Film-Noir', 1), ('Documentary', 1)]

Community 0 (80 movies):
[('Drama', 37), ('Comedy', 28), ('Crime', 24), ('Action', 17), ('Thriller', 16), ('Adventure', 14), ('Mystery', 11), ('Sci-Fi', 11), ('Romance', 11), ('Fantasy', 10), ('War', 8), ('Children', 6), ('Horror', 6), ('Animation', 3), ('Musical', 2), ('Film-Noir', 2), ('Western', 1)]

Community 3 (96 movies):
[('Action', 43), ('Drama', 41), ('Adventure', 41), ('Thriller', 32), ('Sci-Fi', 25), ('Crime', 24), ('Comedy', 22), ('Fantasy', 20), ('IMAX', 16), ('Romance', 14), ('Animation', 13), ('Mystery', 12), ('Children', 12), ('War', 4), ('Horror', 3), ('Musical', 1), ('Film-Noir', 1), ('Western', 1)]

Community 2 (56 movies):
[('Action', 

Results

In [ ]:
for community_id, genre_counts in community_genres.items():
    total_movies = sum(genre_counts.values())
    if total_movies > 0:
        most_common_genre, count = genre_counts.most_common(1)[0]
        purity = count / total_movies
        print(f"Community {community_id}: Most common genre = {most_common_genre} ({purity:.2%})")

Community 1: Most common genre = Drama (19.78%)
Community 0: Most common genre = Drama (17.87%)
Community 3: Most common genre = Action (13.23%)
Community 2: Most common genre = Action (17.95%)
Community 4: Most common genre = Adventure (20.00%)


Analyzing genres in communties based on inference class

In [ ]:
from collections import Counter

# Function to split genres and count them per community
def analyze_genres_per_community(df):
    # Create a dictionary to store genre counts per community
    community_genres = {}

    # Iterate over each community
    for community_id in df['stat_inf_class'].unique():
        # Filter movies in this community
        community_movies = df[df['stat_inf_class'] == community_id]
        # Split genres and flatten the list
        genres_list = []
        for genres in community_movies['genres']:
            if pd.notna(genres) and genres != '(no genres listed)':
                genres_list.extend(genres.split('|'))
        # Count genre occurrences
        genre_counts = Counter(genres_list)
        community_genres[community_id] = genre_counts

    return community_genres

# Run the analysis
community_genres = analyze_genres_per_community(merged_df)

# Print results
for community_id, genre_counts in community_genres.items():
    print(f"Community {community_id} ({len(merged_df[merged_df['stat_inf_class'] == community_id])} movies):")
    print(genre_counts.most_common())
    print()

Community 8 (1 movies):
[('Adventure', 1), ('Animation', 1), ('Children', 1), ('Comedy', 1), ('Fantasy', 1)]

Community 21 (15 movies):
[('Comedy', 9), ('Children', 6), ('Fantasy', 6), ('Musical', 6), ('Drama', 4), ('Adventure', 3), ('Romance', 3), ('Animation', 2), ('Action', 1), ('Crime', 1)]

Community 29 (2 movies):
[('Action', 2), ('Crime', 2), ('Thriller', 2), ('Drama', 1)]

Community 9 (8 movies):
[('Thriller', 6), ('Drama', 5), ('Action', 4), ('Adventure', 4), ('Romance', 2), ('Comedy', 1), ('IMAX', 1), ('War', 1), ('Crime', 1), ('Sci-Fi', 1)]

Community 28 (3 movies):
[('Drama', 2), ('Crime', 1), ('Action', 1), ('Comedy', 1), ('Musical', 1), ('War', 1)]

Community 14 (12 movies):
[('Comedy', 5), ('Thriller', 5), ('Drama', 5), ('Romance', 5), ('Action', 4), ('Sci-Fi', 2), ('Fantasy', 2), ('Crime', 1), ('Adventure', 1), ('Western', 1), ('Animation', 1), ('Children', 1), ('Musical', 1), ('IMAX', 1)]

Community 1 (4 movies):
[('Thriller', 2), ('Drama', 1), ('Romance', 1), ('Myster

Broader classification of genres

In [ ]:
#inf class results
import pandas as pd
from collections import Counter

genre_mapping = {
    'Action': 'Action',
    'Adventure': 'Action',
    'Sci-Fi': 'Thriller',
    'Thriller': 'Thriller',
    'War': 'Action',
    'Comedy': 'Comedy',
    'Children\'s': 'Animation',
    'Animation': 'Animation',
    'Musical': 'Romance',
    'Romance': 'Romance',
    'Drama': 'Drama',
    'Crime': 'Crime',
    'Mystery': 'Thriller',
    'Film-Noir': 'Crime',
    'Western': 'Drama',
    'Fantasy': 'Animation',
    'Horror': 'Horror',
    'Documentary': 'Other',
    '(no genres listed)': 'Other'
}

# Load and merge data
nodes_df = pd.read_csv('community_detection.csv')
nodes_df['movieId'] = nodes_df['Id']
movies_df = pd.read_csv('movies.csv')
movies_df['movieId'] = movies_df['movieId'].astype(int)
merged_df = pd.merge(nodes_df, movies_df, on='movieId', how='inner')
merged_df.to_csv('movies_with_communities.csv', index=False)

# Function to map genres to broad categories and analyze per community
def analyze_broad_genres_per_community(df):
    community_genres = {}

    for community_id in df['stat_inf_class'].unique():
        community_movies = df[df['stat_inf_class'] == community_id]
        broad_genres_list = []

        # Process each movie's genres
        for genres in community_movies['genres']:
            if pd.notna(genres) and genres != '(no genres listed)':
                sub_genres = genres.split('|')
                # Map each sub-genre to its broad category
                for sub_genre in sub_genres:
                    broad_genre = genre_mapping.get(sub_genre, 'Other')
                    broad_genres_list.append(broad_genre)
            else:
                broad_genres_list.append('Other')

        # Count broad genres
        genre_counts = Counter(broad_genres_list)
        community_genres[community_id] = genre_counts

    return community_genres

# Run analysis
community_genres = analyze_broad_genres_per_community(merged_df)

# Print results
for community_id, genre_counts in community_genres.items():
    total_genres = sum(genre_counts.values())
    print(f"Community {community_id} ({len(merged_df[merged_df['stat_inf_class'] == community_id])} movies):")
    for genre, count in genre_counts.most_common():
        print(f"  {genre}: {count} ({count/total_genres:.2%})")
    print()

# Verify the merge
print("Merged Data Preview:")
print(merged_df[['movieId', 'title', 'genres', 'stat_inf_class']].head())

Community 8 (1 movies):
  Animation: 2 (40.00%)
  Action: 1 (20.00%)
  Other: 1 (20.00%)
  Comedy: 1 (20.00%)

Community 21 (15 movies):
  Comedy: 9 (21.95%)
  Romance: 9 (21.95%)
  Animation: 8 (19.51%)
  Other: 6 (14.63%)
  Action: 4 (9.76%)
  Drama: 4 (9.76%)
  Crime: 1 (2.44%)

Community 29 (2 movies):
  Action: 2 (28.57%)
  Crime: 2 (28.57%)
  Thriller: 2 (28.57%)
  Drama: 1 (14.29%)

Community 9 (8 movies):
  Action: 9 (34.62%)
  Thriller: 7 (26.92%)
  Drama: 5 (19.23%)
  Romance: 2 (7.69%)
  Comedy: 1 (3.85%)
  Other: 1 (3.85%)
  Crime: 1 (3.85%)

Community 28 (3 movies):
  Drama: 2 (28.57%)
  Action: 2 (28.57%)
  Crime: 1 (14.29%)
  Comedy: 1 (14.29%)
  Romance: 1 (14.29%)

Community 14 (12 movies):
  Thriller: 7 (20.00%)
  Drama: 6 (17.14%)
  Romance: 6 (17.14%)
  Comedy: 5 (14.29%)
  Action: 5 (14.29%)
  Animation: 3 (8.57%)
  Other: 2 (5.71%)
  Crime: 1 (2.86%)

Community 1 (4 movies):
  Thriller: 4 (44.44%)
  Action: 2 (22.22%)
  Drama: 1 (11.11%)
  Romance: 1 (11.11%)
  Co

In [ ]:
# modularity_class results
nodes_df = pd.read_csv('community_detection.csv')
nodes_df['movieId'] = nodes_df['Id']
movies_df = pd.read_csv('movies.csv')
movies_df['movieId'] = movies_df['movieId'].astype(int)
merged_df = pd.merge(nodes_df, movies_df, on='movieId', how='inner')
merged_df.to_csv('movies_with_communities.csv', index=False)

# Function to map genres to broad categories and analyze per community
def analyze_broad_genres_per_community(df):
    community_genres = {}

    for community_id in df['modularity_class'].unique():
        community_movies = df[df['modularity_class'] == community_id]
        broad_genres_list = []

        # Process each movie's genres
        for genres in community_movies['genres']:
            if pd.notna(genres) and genres != '(no genres listed)':
                sub_genres = genres.split('|')
                # Map each sub-genre to its broad category
                for sub_genre in sub_genres:
                    broad_genre = genre_mapping.get(sub_genre, 'Other')
                    broad_genres_list.append(broad_genre)
            else:
                broad_genres_list.append('Other')

        # Count broad genres
        genre_counts = Counter(broad_genres_list)
        community_genres[community_id] = genre_counts

    return community_genres

# Run analysis
community_genres = analyze_broad_genres_per_community(merged_df)

# Print results
for community_id, genre_counts in community_genres.items():
    total_genres = sum(genre_counts.values())
    print(f"Community {community_id} ({len(merged_df[merged_df['modularity_class'] == community_id])} movies):")
    for genre, count in genre_counts.most_common():
        print(f"  {genre}: {count} ({count/total_genres:.2%})")
    print()

# Verify the merge
print("Merged Data Preview:")
print(merged_df[['movieId', 'title', 'genres', 'modularity_class']].head())

Community 1 (99 movies):
  Action/Adventure: 95 (34.17%)
  Drama/Serious: 88 (31.65%)
  Comedy/Light: 63 (22.66%)
  Other: 15 (5.40%)
  Fantasy/Imaginative: 11 (3.96%)
  Horror: 6 (2.16%)

Community 0 (80 movies):
  Drama/Serious: 75 (36.23%)
  Action/Adventure: 66 (31.88%)
  Comedy/Light: 44 (21.26%)
  Fantasy/Imaginative: 10 (4.83%)
  Other: 6 (2.90%)
  Horror: 6 (2.90%)

Community 3 (96 movies):
  Action/Adventure: 145 (44.62%)
  Drama/Serious: 79 (24.31%)
  Comedy/Light: 50 (15.38%)
  Other: 28 (8.62%)
  Fantasy/Imaginative: 20 (6.15%)
  Horror: 3 (0.92%)

Community 2 (56 movies):
  Action/Adventure: 90 (57.69%)
  Comedy/Light: 32 (20.51%)
  Drama/Serious: 27 (17.31%)
  Other: 4 (2.56%)
  Fantasy/Imaginative: 3 (1.92%)

Community 4 (3 movies):
  Action/Adventure: 5 (33.33%)
  Fantasy/Imaginative: 3 (20.00%)
  Drama/Serious: 3 (20.00%)
  Other: 3 (20.00%)
  Comedy/Light: 1 (6.67%)

Merged Data Preview:
   movieId                           title  \
0        1                Toy Story